# 01 — Ingest from Basketball Reference (team game logs)

Scrape one **team season game-log** page per team per season and land it as raw JSON in the
bronze **landing Volume**. Notebook `02` picks these files up with Auto Loader.

**Why game logs instead of per-day box scores?** Basketball Reference rate-limits to
~20 requests/min and **IP-blocks you for ~1 hour** if you exceed it. The per-day box-score
endpoint needs ~170 requests *per season*. A team game-log page —
`/teams/{ABBR}/{season_end_year}/gamelog/` — returns **every game's full box score for that
team and its opponent in a single request**, so the whole demo is just
**30 teams × N seasons ≈ 90 requests**. That keeps the Four Factors model *and* is friendly
enough to re-run.

| Field group | Source on the page | Role |
|-------------|--------------------|------|
| date / home-away / opponent / W-L / points | gamelog row | **labels** (who won) |
| team & opponent FG/3P/FT/REB/TOV… | gamelog row (`tgl_basic`) | **features** (Four Factors) |
| regular vs playoff | `team_game_log_reg` vs `team_game_log_post` table | filtering |

**Well-Architected Framework notes**

- **Reliability** — already-landed team files are skipped, so runs are idempotent/resumable.
- **Cost / politeness** — `NBA_SCRAPE_THROTTLE_SECONDS` (≥5s) keeps us ~12 req/min; we
  honor `Retry-After` on any 429. `NBA_MAX_TEAMS` ingests a tiny slice for a fast demo.
- **Schema-on-read** — we land *every* cell from the table (keyed by Basketball Reference's
  `data-stat`) as raw VARIANT; the silver layer selects and types only what it needs.

In [1]:
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json, time
import requests
import lxml.html

from nba_helpers import CURRENT_TEAM_ABBREVIATIONS

load_dotenv()
w = WorkspaceClient()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_data"

SEASONS   = [int(s) for s in os.getenv("NBA_SEASONS", "2023,2024,2025").split(",") if s.strip()]
THROTTLE  = float(os.getenv("NBA_SCRAPE_THROTTLE_SECONDS", "5.0"))
_max      = os.getenv("NBA_MAX_TEAMS", "").strip()
MAX_TEAMS = int(_max) if _max else None

TEAMS = CURRENT_TEAM_ABBREVIATIONS[:MAX_TEAMS] if MAX_TEAMS else CURRENT_TEAM_ABBREVIATIONS
UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

print(f"Landing Volume : {VOLUME_PATH}")
print(f"Seasons        : {SEASONS}")
print(f"Teams/season   : {len(TEAMS)}  ({'capped' if MAX_TEAMS else 'all 30'})")
print(f"Throttle       : {THROTTLE}s  (~{60/THROTTLE:.0f} req/min)")
print(f"Total requests : up to {len(SEASONS) * len(TEAMS)} (minus anything already cached)")

Landing Volume : /Volumes/alexander_booth/basketball_reference_waf_bronze/raw_data
Seasons        : [2023, 2024, 2025]
Teams/season   : 30  (all 30)
Throttle       : 5.0s  (~12 req/min)
Total requests : up to 90 (minus anything already cached)


## Helpers — polite fetch, gamelog parser, idempotent landing

The gamelog table is parsed by Basketball Reference's `data-stat` attribute (robust to
column reordering). We strip HTML comments first because BR hides some tables (e.g. the
playoff log) inside `<!-- -->` to deter naive scrapers.

In [2]:
def fetch(url, what, retries=4):
    """GET with throttle-after-success and Retry-After-aware backoff on 429."""
    for attempt in range(retries):
        r = requests.get(url, headers={"User-Agent": UA}, timeout=60)
        if r.status_code == 200:
            time.sleep(THROTTLE)
            return r.text
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "60")) + 2
            print(f"    429 for {what}: honoring Retry-After={wait}s (you exceeded ~20/min)")
            time.sleep(wait)
            continue
        print(f"    HTTP {r.status_code} for {what}; backoff")
        time.sleep(THROTTLE * (2 ** attempt) + 5)
    print(f"    FAILED {what} after {retries} tries")
    return None

def parse_gamelog(html, abbr, year):
    """Return a list of row dicts (every cell keyed by data-stat) for one team-season."""
    html = html.replace("<!--", "").replace("-->", "")  # un-hide commented tables
    doc = lxml.html.fromstring(html)
    rows = []
    for table_id, is_playoff in (("team_game_log_reg", False), ("team_game_log_post", True)):
        try:
            table = doc.get_element_by_id(table_id)
        except KeyError:
            continue
        for tr in table.xpath(".//tbody/tr"):
            if "thead" in (tr.get("class") or ""):   # repeated header rows
                continue
            cells = {}
            for c in tr.xpath("./th|./td"):
                ds = c.get("data-stat")
                if ds:
                    cells[ds] = c.text_content().strip()
            if not cells.get("date"):                # blank/spacer rows
                continue
            cells["team"] = abbr
            cells["season_end_year"] = year
            cells["is_playoff"] = is_playoff
            rows.append(cells)
    return rows

def landed_names(directory):
    try:
        return {f.name for f in w.files.list_directory_contents(directory)}
    except Exception:
        return set()

def upload_json(path, records):
    w.files.upload(path, json.dumps(records).encode("utf-8"), overwrite=True)

print("Helpers ready.")

Helpers ready.


## Scrape every team-season and land it

One request per team-season. Files already in the Volume are skipped.

In [3]:
totals = {"downloaded": 0, "skipped": 0, "failed": 0, "rows": 0}

for year in SEASONS:
    print(f"\n===== Season {year - 1}-{str(year)[2:]} (season_end_year={year}) =====")
    out_dir = f"{VOLUME_PATH}/team_gamelog/season={year}"
    already = landed_names(out_dir)
    for i, abbr in enumerate(TEAMS, 1):
        fname = f"team={abbr}.json"
        if fname in already:
            totals["skipped"] += 1
            continue
        url = f"https://www.basketball-reference.com/teams/{abbr}/{year}/gamelog/"
        html = fetch(url, f"{abbr} {year}")
        if html is None:
            totals["failed"] += 1
            continue
        rows = parse_gamelog(html, abbr, year)
        if not rows:
            print(f"    {abbr}: parsed 0 rows (unexpected) -- skipping write")
            totals["failed"] += 1
            continue
        upload_json(f"{out_dir}/{fname}", rows)
        totals["downloaded"] += 1
        totals["rows"] += len(rows)
        if i % 5 == 0 or i == len(TEAMS):
            print(f"    {i:>2}/{len(TEAMS)} teams "
                  f"(downloaded={totals['downloaded']}, skipped={totals['skipped']}, "
                  f"failed={totals['failed']}, rows={totals['rows']})")

print(f"\nDone. {totals}")


===== Season 2022-23 (season_end_year=2023) =====


     5/30 teams (downloaded=5, skipped=0, failed=0, rows=440)


    10/30 teams (downloaded=10, skipped=0, failed=0, rows=888)


    15/30 teams (downloaded=15, skipped=0, failed=0, rows=1325)


    20/30 teams (downloaded=20, skipped=0, failed=0, rows=1779)


...

    10/30 teams (downloaded=70, skipped=0, failed=0, rows=6124)


    15/30 teams (downloaded=75, skipped=0, failed=0, rows=6580)


    20/30 teams (downloaded=80, skipped=0, failed=0, rows=7032)


    25/30 teams (downloaded=85, skipped=0, failed=0, rows=7470)


    30/30 teams (downloaded=90, skipped=0, failed=0, rows=7880)

Done. {'downloaded': 90, 'skipped': 0, 'failed': 0, 'rows': 7880}


## Verify what landed (and peek at the raw fields)

Confirms file counts and shows the `data-stat` keys we captured — handy when wiring up the
silver layer.

In [4]:
for year in SEASONS:
    files = landed_names(f"{VOLUME_PATH}/team_gamelog/season={year}")
    print(f"season {year}: {len(files)} team files")

# Peek at one record so we can see the exact field names available downstream.
sample_year = SEASONS[0]
sample_dir = f"{VOLUME_PATH}/team_gamelog/season={sample_year}"
sample_files = sorted(landed_names(sample_dir))
if sample_files:
    rec = json.loads(w.files.download(f"{sample_dir}/{sample_files[0]}").contents.read())[0]
    print(f"\nSample record from {sample_files[0]} ({len(rec)} fields):")
    for k in sorted(rec):
        print(f"  {k!r}: {rec[k]!r}")

season 2023: 30 team files


season 2024: 30 team files


season 2025: 30 team files



Sample record from team=ATL.json (54 fields):
  'ast': '30'
  'blk': '5'
  'date': '2022-10-19'
  'drb': '34'
  'efg_pct': '.539'
  'fg': '45'
  'fg2': '38'
  'fg2_pct': '.585'
  'fg2a': '65'
  'fg3': '7'
  'fg3_pct': '.280'
  'fg3a': '25'
  'fg_pct': '.500'
  'fga': '90'
  'ft': '20'
  'ft_pct': '.833'
  'fta': '24'
  'game_location': ''
  'is_playoff': False
  'opp_ast': '25'
  'opp_blk': '3'
  'opp_drb': '39'
  'opp_efg_pct': '.474'
  'opp_fg': '42'
  'opp_fg2': '33'
  'opp_fg2_pct': '.524'
  'opp_fg2a': '63'
  'opp_fg3': '9'
  'opp_fg3_pct': '.257'
  'opp_fg3a': '35'
  'opp_fg_pct': '.429'
  'opp_fga': '98'
  'opp_ft': '14'
  'opp_ft_pct': '.933'
  'opp_fta': '15'
  'opp_name_abbr': 'HOU'
  'opp_orb': '15'
  'opp_pf': '20'
  'opp_stl': '4'
  'opp_team_game_score': '107'
  'opp_tov': '16'
  'opp_trb': '54'
  'orb': '4'
  'overtimes': ''
  'pf': '18'
  'ranker': '1'
  'season_end_year': 2023
  'stl': '12'
  'team': 'ATL'
  'team_game_num_season': '1'
  'team_game_result': 'W'
  'tea

Raw gamelog JSON is in the landing Volume. **Next:** `02_bronze_autoloader` ingests it into
a VARIANT Delta table with Auto Loader.